# Monthly European Electricity Generation

## Goal

Build a reproducible monthly time series from the bundled Eurostat exports, validate its units and coverage, visualise a selected-category generation series, decompose its seasonality, test stationarity, and evaluate a chronological forecast baseline.

**Source definition.** The workbooks contain **net electricity generation by fuel, monthly, in gigawatt-hours (GWh)**. They are not energy-consumption data and they are not measured in terajoules. The export was updated on 22 August 2024 and covers January 2016 through July 2024; the final year is partial.

Source: [Eurostat energy database](https://ec.europa.eu/eurostat/web/energy/database). The bundled files make this notebook reproducible but should be refreshed before using the results as current statistics.

**Accounting limits:** This tutorial selects combustible renewables, geothermal, broad hydro, other renewables, solar and wind. The broad Hydro export does not separate pumped-storage output; the selected sum must not be called a verified renewable-only total. Eurostat distinguishes storage generation from renewable primary production. The historical non-renewable combustible workbook has an unverified category boundary and is not presented as a complete total. Reporting-country coverage changes, flags must be inspected, and some values are missing. The loader recognizes `:` as missing and stops on unexpected text or non-finite measurements so those values are not silently discarded. Unflagged zeros do not prove complete reporting; in particular, check solar coverage against national sources and comparable definitions. This is not an EU-27 official total. For a renewable-only analysis obtain the required hydro breakdown and consistent country coverage first. [Eurostat electricity statistics](https://ec.europa.eu/eurostat/statistics-explained/SEPDF/cache/9990.pdf).

## Setup

The notebook uses pandas, NumPy, Matplotlib, and statsmodels. Paths are resolved whether execution starts in the course root or beside the notebook.

In [ ]:
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 12)

def find_course_root(start=Path.cwd()):
    for candidate in [start / "EAGE_PythonRenewableEnergyCourse", start, *start.parents]:
        if (candidate / "myst.yml").exists() and (candidate / "data").exists():
            return candidate
    raise FileNotFoundError("Run this notebook from inside the course repository.")

COURSE_ROOT = find_course_root()
DATA_DIR = COURSE_ROOT / "data" / "section4" / "euroStat"
DATA_DIR

## Data

The shorthand `oil` represents oil and petroleum products, excluding the biofuel portion, as stated in the bundled workbook. The historical combustible-fuel workbook is labelled `historical_nonrenewable_unverified`: its original category boundary has not been confirmed. Do not treat it as a complete non-renewable total or add it to other categories without verifying their definitions and overlap. The [Eurostat SIEC vocabulary](https://dd.eionet.europa.eu/vocabulary/eurostat/siec/view?page=2) lists both `CF_NR` and `CF_NR_OTH` with an accepted date of 7 December 2018, but their concept entries provide no definitions. This does not identify the code selected in the historical workbook or establish the relationship between the two series.

In [ ]:
SOURCE_FILES = {
    "coal_and_manufactured_gases": "nrg_cb_pem_page_spreadsheet_coal.xlsx",
    "combustible_renewable": "nrg_cb_pem_page_spreadsheet_combustionFuels_Renewables.xlsx",
    "historical_nonrenewable_unverified": "nrg_cb_pem_page_spreadsheet_combustionFuels_nonRenewables.xlsx",
    "geothermal": "nrg_cb_pem_page_spreadsheet_geothermal.xlsx",
    "hydro": "nrg_cb_pem_page_spreadsheet_hydro.xlsx",
    "natural_gas": "nrg_cb_pem_page_spreadsheet_naturalGas.xlsx",
    "nuclear_and_other_fuels_nec": "nrg_cb_pem_page_spreadsheet_nuclear.xlsx",
    "oil": "nrg_cb_pem_page_spreadsheet_oil.xlsx",
    "other_renewable": "nrg_cb_pem_page_spreadsheet_otherRenewables.xlsx",
    "solar": "nrg_cb_pem_page_spreadsheet_solar.xlsx",
    "wind": "nrg_cb_pem_page_spreadsheet_wind.xlsx",
}

SELECTED_SOURCES = [
    "combustible_renewable", "geothermal", "hydro",
    "other_renewable", "solar", "wind",
]

def load_eurostat_export(path: Path, source: str) -> pd.DataFrame:
    with pd.ExcelFile(path) as workbook:
        wide = pd.read_excel(workbook, sheet_name=workbook.sheet_names[-1], skiprows=8)
    # Every month has an adjacent status column, including flags for missing values.
    columns = list(wide.columns)
    month_columns = [c for c in columns if not str(c).startswith("Unnamed") and c != columns[0]]
    flag_wide = pd.DataFrame({"country": wide[columns[0]]})
    for month in month_columns:
        next_index = columns.index(month) + 1
        flag_wide[month] = (wide[columns[next_index]] if next_index < len(columns)
                            and str(columns[next_index]).startswith("Unnamed") else pd.NA)
    wide = wide[[columns[0], *month_columns]].rename(columns={columns[0]: "country"})
    wide = wide.iloc[1:].copy()  # remove GEO label row
    footer = wide.index[wide["country"].eq("Special value")]
    if len(footer) != 1:
        raise ValueError("Workbook footer changed; inspect the source layout")
    wide = wide.loc[wide.index < footer[0]]
    wide = wide.dropna(subset=["country"])
    wide = wide[~wide["country"].astype(str).str.startswith("European Union")]
    long = wide.melt(id_vars="country", var_name="month", value_name="generation_gwh")
    # Keep status rows aligned with retained country rows, including after blank rows.
    flag_long = flag_wide.loc[wide.index].melt(id_vars="country", var_name="month", value_name="status_flag")
    long = long.merge(flag_long, on=["country", "month"], how="left", validate="one_to_one")
    long["month"] = pd.to_datetime(long["month"], format="%Y-%m")
    long["generation_gwh"] = pd.to_numeric(
        long["generation_gwh"].replace({":": np.nan}), errors="raise"
    )
    if not np.isfinite(long["generation_gwh"].dropna()).all():
        raise ValueError("Generation values must be finite or missing; inspect the source workbook")
    long["source"] = source
    return long[["country", "month", "source", "generation_gwh", "status_flag"]]

generation = pd.concat(
    [load_eurostat_export(DATA_DIR / filename, source)
     for source, filename in SOURCE_FILES.items()],
    ignore_index=True,
)
generation.head()

### Checks

The intended grain is one country–month–source row. Missing values remain missing; they are never silently replaced by zero.

In [ ]:
key = ["country", "month", "source"]
if generation.duplicated(key).any():
    raise ValueError("Duplicate country/month/source observations")
if generation["month"].min() != pd.Timestamp("2016-01-01") or generation["month"].max() != pd.Timestamp("2024-07-01"):
    raise ValueError("Unexpected historical snapshot month range")
negative_rows = generation["generation_gwh"].lt(0).sum()

# Source-file country sets are coverage diagnostics, not proof of valid measurements.
def country_coverage(frame):
    country_sets = frame.groupby("source")["country"].agg(lambda values: set(values.dropna()))
    union = set().union(*country_sets)
    return pd.DataFrame([
        {"source": source, "country_count": len(countries),
         "countries": ", ".join(sorted(countries)),
         "absent_from_source": ", ".join(sorted(union - countries))}
        for source, countries in country_sets.items()
    ]).set_index("source")

coverage_by_source = country_coverage(generation)
print("Source-file country coverage (differences need review):")
print(coverage_by_source[["country_count", "absent_from_source"]].to_string())
print("Countries across all source files:", ", ".join(sorted(generation["country"].dropna().unique())))

quality = pd.Series({
    "rows": len(generation),
    "countries": generation["country"].nunique(),
    "months": generation["month"].nunique(),
    "sources": generation["source"].nunique(),
    "missing_share": generation["generation_gwh"].isna().mean(),
    "negative_rows_to_review": negative_rows,
})
quality

## Results

For a regional total, sum reported countries and retain a coverage count. A lower total can otherwise mean either lower generation or fewer reporting countries.

In [ ]:
monthly_source = (
    generation.groupby(["month", "source"], as_index=False)
    .agg(
        generation_gwh=("generation_gwh", lambda values: values.sum(min_count=1)),
        reporting_countries=("generation_gwh", "count"),
    )
)

selected_monthly = (
    monthly_source[monthly_source["source"].isin(SELECTED_SOURCES)]
    .groupby("month")
    .agg(
        selected_generation_gwh=("generation_gwh", lambda values: values.sum(min_count=len(SELECTED_SOURCES))),
        source_categories_reported=("generation_gwh", "count"),
    )
)
selected_monthly.tail(8)

In [ ]:
coverage = monthly_source[monthly_source["source"].isin(SELECTED_SOURCES)].pivot(
    index="month", columns="source", values="reporting_countries")
ax = coverage.plot(figsize=(11, 5))
ax.set(title="Reporting coverage changes: do not interpret raw sums as trends",
       xlabel="Month", ylabel="Countries reporting a numeric value")
ax.legend(title="Source", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()

### Seasonality and trend

Regional reporting coverage changes, so we use a single country and source: Austrian wind generation. We trim missing endpoints and reject internal gaps instead of filling them. The series is monthly, so one annual seasonal cycle has period 12. `seasonal_decompose` requires at least two complete cycles. The final 2024 year is partial and must not be compared with complete annual totals.

In [ ]:
country_series = generation.loc[
    generation["country"].eq("Austria") & generation["source"].eq("wind")
].set_index("month")["generation_gwh"].sort_index()
series = country_series.loc[country_series.first_valid_index():country_series.last_valid_index()].asfreq("MS")
print(f"Austria wind: {series.index.min():%Y-%m} to {series.index.max():%Y-%m}; {len(series)} months")
assert len(series) >= 24 and series.notna().all()

decomposition = seasonal_decompose(series, model="additive", period=12, extrapolate_trend="freq")
fig = decomposition.plot()
fig.set_size_inches(11, 8)
fig.suptitle("Austria wind generation: additive decomposition", y=1.01)
plt.tight_layout()

### Stationarity

The Augmented Dickey–Fuller test evaluates a unit-root null hypothesis. It is not a seasonality test. A small p-value is evidence against a unit root under the selected deterministic terms and lag rule; it does not prove that every form of non-stationarity is absent.

In [ ]:
adf_stat, p_value, used_lag, n_obs, critical, _ = adfuller(series, autolag="AIC")
pd.Series({
    "ADF statistic": adf_stat,
    "p-value": p_value,
    "lags used": used_lag,
    "observations": n_obs,
    **{f"critical value {level}": value for level, value in critical.items()},
})

### Chronological forecast baseline

The last 12 complete observations form the test window. A seasonal-naïve forecast uses the corresponding month one year earlier. This baseline is simple, auditable, and contains no future leakage.

In [ ]:
test_horizon = 12
train = series.iloc[:-test_horizon]
test = series.iloc[-test_horizon:]
seasonal_naive = series.shift(12).reindex(test.index)

mae = (test - seasonal_naive).abs().mean()
rmse = np.sqrt(((test - seasonal_naive) ** 2).mean())

ax = series.iloc[-36:].plot(figsize=(11, 4), label="Observed", color="black")
seasonal_naive.plot(ax=ax, label="Seasonal-naïve forecast", color="tab:orange")
ax.axvline(test.index.min(), color="tab:red", linestyle="--", label="Test starts")
ax.set(title=f"Austria wind baseline — MAE {mae:,.0f} GWh; RMSE {rmse:,.0f} GWh",
       xlabel="Month", ylabel="Generation (GWh)")
ax.legend()
plt.tight_layout()
pd.Series({"MAE_GWh": mae, "RMSE_GWh": rmse})

## Takeaways

- Preserve the source definition: generation, not consumption; GWh, not TJ.
- Never add an aggregate fuel category to its components.
- Retain missing values and report coverage alongside regional totals.
- Use a 12-month seasonal period for monthly data.
- Evaluate forecasts on future time windows and compare them with a seasonal-naïve baseline.
- Refresh the Eurostat export before making current-policy or market claims.